# 06b — Full Supervised Baseline — ELECTRA-small

Trains `utils.config.CLASSIFIER_MODEL_NAME_ALT` on 100% of the train split
(full data, `config.CLASSIFIER_SAMPLE_SIZE=None`) — the upper-bound
reference every other method is compared against, 16-way.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_semisupervised
from utils.modeling import get_predictions, train_model
from utils.samples import save_full_output, save_label_samples

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

train_sample = stratified_sample(train_clean, config.CLASSIFIER_SAMPLE_SIZE, seed=config.SEED)
print(f"Training on {len(train_sample)} fully-labeled rows (upper bound baseline)")

Training on 319 fully-labeled rows (upper bound baseline)


In [3]:
model, tokenizer = train_model(train_sample, model_name=config.CLASSIFIER_MODEL_NAME_ALT, epochs=3)

test_probs = get_predictions(model, tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_full_supervised_electra.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_full_supervised_electra.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved full-supervised baseline results.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,2.763808


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0

                precision    recall  f1-score   support

ARTS & CULTURE       0.00      0.00      0.00         5
      BUSINESS       0.00      0.00      0.00         5
        COMEDY       0.00      0.00      0.00         5
         CRIME       0.00      0.00      0.00         5
     EDUCATION       0.00      0.00      0.00         5
 ENTERTAINMENT       0.00      0.00      0.00         5
   ENVIRONMENT       0.00      0.00      0.00         5
        HEALTH       0.00      0.00      0.00         5
         MEDIA       0.00      0.00      0.00         5
          NEWS       0.00      0.00      0.00         5
      POLITICS       0.00      0.00      0.00         5
      RELIGION       0.00      0.00      0.00         5
       SCIENCE       0.07      1.00      0.13         5
        SPORTS       0.25      0.20      0.22         5
          TECH       0.00      0.00      0.00         5
         WOMEN       0.00      0.00      0.00         5

      accuracy                           0.07 

In [4]:
save_label_samples(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(),
    config.CLASS_NAMES, confidence=test_probs.max(axis=1), n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_full_supervised_electra.csv")
save_full_output(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(), config.CLASS_NAMES,
    confidence=test_probs.max(axis=1), extra_columns={"summary": test_clean["summary"].tolist()},
    path=config.RESULTS_DIR / "full_labels_full_supervised_electra.csv")
print("Saved sample + full-row outputs for full_supervised_electra.")

Saved sample + full-row outputs for full_supervised_electra.
